In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
import time, torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Encoder, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)

    def forward(self, x):
        outputs, (hidden, cell) = self.rnn(x)
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, encoder_outputs, decoder_hidden):
        decoder_hidden = decoder_hidden[-1].unsqueeze(1).repeat(1, encoder_outputs.size(1), 1)
        concat = torch.cat((encoder_outputs, decoder_hidden), dim=2)
        energy = self.attn(concat).squeeze(2)
        weights = torch.softmax(energy, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context

# =========  Decoder  =========
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size=1):
        super().__init__()
        self.attention = Attention(hidden_size)
        self.rnn       = nn.LSTM(hidden_size + 1, hidden_size, batch_first=True)
        self.fc        = nn.Linear(hidden_size, output_size)

    def forward(self, y_prev, hidden, cell, enc_out):
        """
        y_prev 允许是 (batch,) / (batch,1) / (batch,1,1)
        返回   : (batch,1) — 预测一个时间步
        """
        # ---- 断言确保 y_prev 不是带 6 那条维度 ----
        y_prev = y_prev.squeeze()                 # → (batch,)
        assert y_prev.ndim == 1, f"expect (batch,) got {y_prev.shape}"

        # ---- reshape & Attention ----
        y_prev  = y_prev.view(-1, 1, 1)           # → (batch,1,1)
        context = self.attention(enc_out, hidden) # → (batch,hidden)
        context = context.unsqueeze(1)            # → (batch,1,hidden)

        # ---- 拼接后送入 Decoder LSTM ----
        rnn_in  = torch.cat((y_prev, context), dim=2)   # (batch,1,hidden+1)
        out, (hidden, cell) = self.rnn(rnn_in, (hidden, cell))
        pred = self.fc(out.squeeze(1))                  # (batch,1)
        return pred, hidden, cell


class DARNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, output_size=1):
        super(DARNN, self).__init__()
        self.encoder = Encoder(input_size, hidden_size)
        self.decoder = Decoder(hidden_size, output_size)

    def forward(self, x, y_init):
    # x: (batch, input_len, 1)
    # y_init: (batch, pred_len)  —— 这里 pred_len 就是 6
        assert y_init.ndim == 2, f"y_init expects (batch,pred_len), got {y_init.shape}"
        encoder_outputs, hidden, cell = self.encoder(x)
        outputs = []
        for t in range(y_init.size(1)):      # 这里循环 6 次
            y_prev = y_init[:, t]            # (batch, 1)
            out, hidden, cell = self.decoder(y_prev, hidden, cell, encoder_outputs)
            outputs.append(out.unsqueeze(1))
        return torch.cat(outputs, dim=1)     # 输出 (batch, 6, 1)


In [ ]:
# -----------------------------
#  Training / evaluation helper
# -----------------------------
def run_darnn_forecast(X_train, y_train, X_val, y_val, X_test,  y_test, params):

    """Train & evaluate DARNN.

    Parameters
    ----------
    X_train, y_train, X_val, y_val, X_test, y_test : np.ndarray
        Shapes expected:
        - X : (n_samples, input_len, n_features)
        - y : (n_samples, pred_len)
    params : dict
        EXPECTED KEYS ⇒
        HIDDEN_SIZE, LEARNING_RATE, EPOCHS, PATIENCE,
        (optional) DEVICE, PRINT_EVERY

    Returns
    -------
    y_true : np.ndarray
    preds  : np.ndarray
    val_losses : list[float]
    mae, rmse  : float, float
    """
    # --- device & model
    DEVICE = params.get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
    Xva = torch.from_numpy(X_val).float().to(DEVICE)
    Yva = torch.from_numpy(y_val).float().to(DEVICE)

    model = DARNN(input_size=X_train.shape[2],
                  hidden_size=params['HIDDEN_SIZE'],
                  output_size=1).to(DEVICE)

    optimiser = torch.optim.Adam(model.parameters(), lr=params['LEARNING_RATE'])
    criterion = nn.MSELoss()

    # convert data only once (float32)
    Xtr = torch.from_numpy(X_train).float().to(DEVICE)
    Ytr = torch.from_numpy(y_train).float().to(DEVICE)
    Xva = torch.from_numpy(X_val).float().to(DEVICE)
    Yva = torch.from_numpy(y_val).float().to(DEVICE)
    Xte = torch.from_numpy(X_test ).float().to(DEVICE)
    Yte = torch.from_numpy(y_test ).float().to(DEVICE)
    # —— 快速自测 ——
    print("DEBUG shapes:")
    print("  Ytr:", Ytr.shape)        # 期望 (N_train, horizon, 1)
    print("  Yva:", Yva.shape)        # 期望 (N_val,   horizon, 1)
    print("  Yte:", Yte.shape)        # 期望 (N_test,  horizon, 1)
        # ----------------- ⚙️① 训练计时开始 -----------------
    start_time = time.time()              # ← 训练总时长计时器
    early_stopped = 0                     # ← 早停标记
    # ---------------------------------------------------
    best_val, wait = float('inf'), 0
    print_every = params.get('PRINT_EVERY', 1)

 # 初始化
    train_losses = []
    val_losses = []

    for epoch in range(1, params['EPOCHS'] + 1):
        # --- training ---
        model.train()
        optimiser.zero_grad()

        y_init  = Ytr.squeeze(-1)        # (batch, horizon)   ← 先 squeeze
        target  = y_init.unsqueeze(-1)      # (batch, horizon,1) ← 仍 3 维
        out = model(Xtr, y_init)                 # (batch, horizon, 1)
        loss = criterion(out, target)
        loss.backward()
        optimiser.step()

        train_losses.append(loss.item())

        # --- validation ---
        model.eval()
        with torch.no_grad():
            y_init_val = Yva.squeeze(-1)
            target_val = y_init_val.unsqueeze(-1)
            val_out = model(Xva, y_init_val)
            val_loss = criterion(val_out, target_val).item()
        val_losses.append(val_loss)

        if epoch % print_every == 0:
            print(f"Epoch {epoch:02d} | trainMSE={loss.item():.4f} | valMSE={val_loss:.4f}")

                # ---------- early-stopping ----------
        if val_loss < best_val:
            best_val, wait = val_loss, 0
            torch.save(model.state_dict(), 'best_darnn.pth')
        else:
            wait += 1
            if wait >= params['PATIENCE']:
                print("Early stopping triggered.")
                early_stopped = 1          # ⚙️② 置早停标记
                break

    final_epoch = epoch                    # ⚙️③ 记录最终 epoch


    # ---- evaluation on best model ----
    model.load_state_dict(torch.load('best_darnn.pth', map_location=DEVICE))
    model.eval()
    with torch.no_grad():
      preds = model(Xte, Yte.squeeze(-1)).squeeze(-1)   # GPU Tensor (batch, horizon)

    y_true = Yte.squeeze(-1)                              # ★ 加这一行，把测试真值取出来

    # ---------- ⚙️④ 统计训练时间 / 显存峰值 ----------
    training_time = round(time.time() - start_time, 4)
    mem_mb = (torch.cuda.max_memory_reserved() / 1e6
              if torch.cuda.is_available() else -1)
    # -----------------------------------------------

    # ② 如果用 torch 版指标（不搬 CPU），改成：
    mae  = torch.mean(torch.abs(y_true - preds)).item()
    rmse = torch.sqrt(torch.mean((y_true - preds) ** 2)).item()

        # ---------- ⚙️⑤ 返 回 值 ----------
    return (y_true, preds, train_losses, val_losses, mae, rmse, training_time, mem_mb, early_stopped, final_epoch)
